# CreativeGAN-Rhythm M1 Max Optimization

This notebook demonstrates the optimizations made for M1 Max machines running macOS Sequoia.

## Key Changes:
1. Updated to TensorFlow 2.x with Apple Silicon support
2. Added Metal GPU acceleration
3. Enabled mixed precision training
4. Updated all Keras imports to use tf.keras
5. Optimized for M1 Max architecture

In [ ]:
# M1 Max Optimized Setup
import sys
import os

# Add rhythm_can to path
sys.path.append('./rhythm_can')

# Import M1 optimization utilities
from rhythm_can.m1_optimization import configure_tensorflow_for_m1, get_optimized_imports, check_m1_compatibility

# Configure TensorFlow for M1 Max
tf = configure_tensorflow_for_m1()

# Get optimized imports
imports = get_optimized_imports()
globals().update(imports)

print("✅ M1 Max optimization complete!")
print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {tf.keras.__version__}")

In [ ]:
# Test Metal GPU availability
print("Available devices:")
for device in tf.config.list_physical_devices():
    print(f"  - {device}")

# Test GPU computation
with tf.device('/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'):
    a = tf.constant([[1.0, 2.0], [3.0, 4.0]])
    b = tf.constant([[1.0, 1.0], [0.0, 1.0]])
    c = tf.matmul(a, b)
    print(f"Matrix multiplication result: {c.numpy()}")
    print(f"Device used: {c.device}")

In [ ]:
# Import remaining dependencies
import numpy as np
import matplotlib.pyplot as plt
from rhythm_can.constants import *
from rhythm_can.utils import *

print("✅ All dependencies loaded successfully")

In [ ]:
# Example: Create a simple GAN generator optimized for M1 Max
def create_optimized_generator(len_input=100, len_seq=32, nb_notes=9):
    """Create a GAN generator optimized for M1 Max"""
    
    # Input layer
    z_input = Input(shape=(len_input,), name='z_input')
    
    # Dense layers with optimal activation for Metal GPU
    x = Dense(256)(z_input)
    x = LeakyReLU(alpha=0.2)(x)
    x = BatchNormalization()(x)
    
    x = Dense(512)(x)
    x = LeakyReLU(alpha=0.2)(x)
    x = BatchNormalization()(x)
    
    # Output layer
    x = Dense(len_seq * nb_notes, activation='sigmoid')(x)
    output = Reshape((len_seq, nb_notes))(x)
    
    model = Model(inputs=z_input, outputs=output, name='optimized_generator')
    
    return model

# Test the generator
generator = create_optimized_generator()
generator.summary()

# Test inference
test_noise = get_noise(1, len_input)
test_output = generator(test_noise)
print(f"✅ Generator test successful. Output shape: {test_output.shape}")

In [ ]:
# Visualize generated rhythm pattern
plot_drum_matrix(test_output.numpy())
print("✅ Visualization test successful")

## Migration Guide for Existing Notebooks

To update existing notebooks for M1 Max compatibility:

### 1. Replace old imports with optimized imports

**OLD:**
```python
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

from keras.layers import Input, Dense, Flatten, Dropout
from keras.optimizers import RMSprop, Adam
import keras.backend as K
from keras.models import Model
```

**NEW:**
```python
import sys
sys.path.append('./rhythm_can')
from rhythm_can.m1_optimization import configure_tensorflow_for_m1, get_optimized_imports

tf = configure_tensorflow_for_m1()
imports = get_optimized_imports()
globals().update(imports)
```

### 2. Update model compilation for mixed precision

**OLD:**
```python
model.compile(optimizer=Adam(lr=0.0002), loss='binary_crossentropy')
```

**NEW:**
```python
optimizer = Adam(learning_rate=0.0002)
model.compile(optimizer=optimizer, loss='binary_crossentropy')
```

### 3. Update TensorBoard logging

**OLD:**
```python
from tensorboard_logger import configure, log_value
configure(logdir)
log_value('loss', loss_value, step)
```

**NEW:**
```python
writer = start_tfboard_log()
log_value(writer, 'loss', loss_value, step)
```